# Route-3 7D Rescue Preregistration

This notebook displays the machine-locked rescue contract created before any new
scientific simulation. Parameters, prior, fixed `c_ctx2th`, 14D features,
simulator settings, numerical gates, and verdict logic remain unchanged.


In [1]:
from pathlib import Path
import json, os, sys
import numpy as np
import pandas as pd
from IPython.display import display, Image, Markdown

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "S4_sbi").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "S4_sbi").exists():
    raise RuntimeError("sleep_loop project root could not be resolved")
sys.path.insert(0, str(PROJECT_ROOT / "S4_sbi" / "src"))
RESULTS = PROJECT_ROOT / "S4_sbi" / "results" / "route3_7d_rescue"
print("CONDA_DEFAULT_ENV=", os.environ.get("CONDA_DEFAULT_ENV"))
print("sys.executable=", sys.executable)
print("sys.prefix=", sys.prefix)
print("PROJECT_ROOT=", PROJECT_ROOT)
assert "neurolib" in sys.executable.lower()


CONDA_DEFAULT_ENV= neurolib
sys.executable= C:\Users\YUS190\AppData\Local\anaconda3\envs\neurolib\python.exe
sys.prefix= C:\Users\YUS190\AppData\Local\anaconda3\envs\neurolib
PROJECT_ROOT= D:\Year3_Mao_Projects\sleep_loop


## Lock verification


In [2]:
from sleep_sbi.route3_7d_rescue import (
    read_rescue_preregistration, verify_rescue_preregistration,
    assert_seed_and_theta_disjointness,
)
rescue_hash = verify_rescue_preregistration()
cfg = read_rescue_preregistration()
print("locked rescue SHA-256:", rescue_hash)
display(pd.DataFrame({
    "parameter": cfg["frozen_contract"]["parameter_order"],
    "lower": np.asarray(cfg["frozen_contract"]["prior_bounds"])[:, 0],
    "upper": np.asarray(cfg["frozen_contract"]["prior_bounds"])[:, 1],
}))
assert cfg["frozen_contract"]["feature_dimension"] == 14


locked rescue SHA-256: 01e0b0cb2277340872d95fef5fb5f2c0dd15bdf7104d363bd67fb41f1acd4a02


,parameter,lower,upper
0,mue,3.31075,4.47925
1,mui,2.57295,3.48105
2,b,28.40000,42.60000
3,tauA,998.20000,1853.80000
4,g_LK,0.02000,0.07000
5,g_h,0.03500,0.09500
6,c_th2ctx,0.00000,0.07500


## Allowed bounded rescue and selection rule


In [3]:
display(pd.json_normalize(cfg["allowed_rescue_configurations"]))
display(pd.Series(cfg["training_policy"], name="value").to_frame())
display(Markdown("\n".join(f"{i+1}. {v}" for i, v in enumerate(cfg["selection_rule"]))))


,id,architecture.model,architecture.hidden_features,architecture.num_transforms,architecture.z_score_theta,architecture.z_score_x
0,maf64_t5,maf,64,5,none,none
1,maf128_t8,maf,128,8,none,none


,value
batch_size,128
learning_rate,0.0005
weight_decay,0.000001
max_epochs,300
early_stopping_patience,25
minimum_improvement,0.0001
clip_max_norm,5.0
members,5
proposal,same independent uniform 7D prior
rounds,1


1. prefer a raw pipeline satisfying all unchanged calibration and contraction requirements
2. then minimize aggregate absolute coverage error at 50%, 80%, and 90%
3. then minimize marginal CRPS
4. then prefer simpler architecture
5. then lower compute cost

## Independence and claim boundary

The additional training, development, and untouched final sequences must be
disjoint from the old bank/test and from one another. A calibrated pipeline is
separately labelled and is not eligible for Formal GO under the original contract.


In [4]:
independence = assert_seed_and_theta_disjointness()
display(pd.Series(independence["seed_intersections"], name="intersection_count").to_frame())
display(pd.Series(independence["exact_theta_overlaps"], name="exact_overlap_count").to_frame())
assert independence["pass"]
assert cfg["calibration"]["formal_go_eligibility"] is False


,intersection_count
additional_a_vs_old,0
additional_b_vs_old,0
development_vs_old,0
final_vs_old,0
additional_a_vs_additional_b,0
additional_a_vs_development,0
additional_a_vs_final,0
additional_b_vs_development,0
additional_b_vs_final,0
development_vs_final,0


,exact_overlap_count
additional_vs_old,0
development_vs_old,0
final_vs_old,0
additional_vs_development,0
additional_vs_final,0
development_vs_final,0
